# MNIST Image Classification — Improved Model

This notebook improves on the baseline CNN with:
- **Deeper architecture** with more filters and residual-style skip connections
- **Batch Normalization** for stable, faster training
- **Dropout** for regularization and generalization
- **Data Augmentation** for robustness to non-ideal input conditions
- **Learning Rate Scheduling** with ReduceLROnPlateau + Cosine Decay
- **Early Stopping** to prevent overfitting
- **Comprehensive evaluation** with confusion matrix and per-class analysis

## 1. Imports & Setup

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {len(tf.config.list_physical_devices('GPU'))}")


## 2. Data Loading & Exploratory Analysis

In [ ]:
# Load the modified MNIST dataset
data = np.load('mnist_modified.npz')
X_train, y_train = data['X_train'], data['y_train']
X_test,  y_test  = data['X_test'],  data['y_test']

print(f"Train set: {X_train.shape}, dtype={X_train.dtype}, range=[{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Test  set: {X_test.shape},  dtype={X_test.dtype},  range=[{X_test.min():.3f},  {X_test.max():.3f}]")
print(f"Labels — Train: {np.unique(y_train)}, Test: {np.unique(y_test)}")

In [ ]:
# Visualise sample images and class distribution
fig = plt.figure(figsize=(18, 8))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# --- Row 0: Sample images per class ---
ax_imgs = fig.add_subplot(gs[0, :])
ax_imgs.axis('off')
ax_imgs.set_title('Sample images (one per class)', fontsize=13, fontweight='bold', pad=10)

num_classes = 10
samples_per_class = 1
inner_gs = gridspec.GridSpecFromSubplotSpec(1, num_classes, subplot_spec=gs[0, :], wspace=0.05)
for cls in range(num_classes):
    idx = np.where(y_train == cls)[0][0]
    ax  = fig.add_subplot(inner_gs[cls])
    ax.imshow(X_train[idx, :, :, 0], cmap='gray')
    ax.set_title(str(cls), fontsize=11)
    ax.axis('off')

# --- Row 1 left: Class distribution ---
ax_dist = fig.add_subplot(gs[1, 0])
unique, counts = np.unique(y_train, return_counts=True)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(unique)))
ax_dist.bar(unique, counts, color=colors, edgecolor='white')
ax_dist.set_xlabel('Class', fontsize=11)
ax_dist.set_ylabel('Count', fontsize=11)
ax_dist.set_title('Training Class Distribution', fontsize=12, fontweight='bold')
ax_dist.set_xticks(unique)

# --- Row 1 right: Pixel intensity histogram ---
ax_hist = fig.add_subplot(gs[1, 1])
ax_hist.hist(X_train.flatten(), bins=50, color='steelblue', edgecolor='none', alpha=0.8)
ax_hist.set_xlabel('Pixel value', fontsize=11)
ax_hist.set_ylabel('Frequency', fontsize=11)
ax_hist.set_title('Pixel Intensity Distribution (Train)', fontsize=12, fontweight='bold')

plt.suptitle('Exploratory Data Analysis — MNIST Modified', fontsize=15, fontweight='bold', y=1.01)
plt.savefig('eda.png', dpi=150, bbox_inches='tight')
plt.show()
print("Class counts:", dict(zip(unique.tolist(), counts.tolist())))

## 3. Preprocessing

### Key improvements over baseline
| Issue | Baseline | Improved |
|---|---|---|
| Normalisation | None explicit | Z-score per-channel (mean/std of training set) |
| Augmentation | None | Random rotation, shift, zoom, horizontal flip |
| Splits | Train/Test only | Train / Val (10%) / Test |

In [ ]:
# ── Normalisation ────────────────────────────────────────────────────────────
# Compute statistics on the training set only to avoid data leakage
train_mean = X_train.mean()
train_std  = X_train.std()  + 1e-8   # avoid div-by-zero

X_train_norm = (X_train - train_mean) / train_std
X_test_norm  = (X_test  - train_mean) / train_std

print(f"After normalisation — mean: {X_train_norm.mean():.4f}, std: {X_train_norm.std():.4f}")

# ── Validation split (stratified) ────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_norm, y_train,
    test_size=0.10,
    random_state=SEED,
    stratify=y_train
)
print(f"Train: {X_tr.shape} | Val: {X_val.shape} | Test: {X_test_norm.shape}")

In [ ]:
# ── Data Augmentation Pipeline ────────────────────────────────────────────────
# Designed to mimic the 'non-ideal conditions' mentioned in the task brief:
#   - slight rotations  (blurry / tilted scans)
#   - small translations (off-centre framing)
#   - zoom              (varying distances)
#   - horizontal flip   (mirrored samples)
#   - Gaussian noise    (sensor noise / low-quality capture)

BATCH_SIZE = 64

augmentation_layer = tf.keras.Sequential([
    layers.RandomRotation(factor=0.08, fill_mode='nearest'),       # ±29°
    layers.RandomTranslation(height_factor=0.08, width_factor=0.08,
                              fill_mode='nearest'),
    layers.RandomZoom(height_factor=(-0.10, 0.10), fill_mode='nearest'),
    layers.RandomFlip('horizontal'),
], name='augmentation')

# Build tf.data pipelines
AUTO = tf.data.AUTOTUNE

def make_dataset(X, y, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X.astype('float32'), y.astype('int32')))
    if shuffle:
        ds = ds.shuffle(len(X), seed=SEED)
    ds = ds.batch(BATCH_SIZE)
    if augment:
        ds = ds.map(lambda x, y: (augmentation_layer(x, training=True), y),
                    num_parallel_calls=AUTO)
    return ds.prefetch(AUTO)

train_ds = make_dataset(X_tr,         y_tr,  augment=True,  shuffle=True)
val_ds   = make_dataset(X_val,        y_val, augment=False, shuffle=False)
test_ds  = make_dataset(X_test_norm,  y_test,augment=False, shuffle=False)

print("Dataset pipelines created.")
print(f"  Train batches : {len(train_ds)}")
print(f"  Val batches   : {len(val_ds)}")
print(f"  Test batches  : {len(test_ds)}")

In [ ]:
# Visualise augmented samples
sample_images, sample_labels = next(iter(train_ds))

fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    img = sample_images[i].numpy()[:, :, 0]
    ax.imshow(img, cmap='gray')
    ax.set_title(int(sample_labels[i]), fontsize=9)
    ax.axis('off')
plt.suptitle('Augmented Training Samples (normalised)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('augmented_samples.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Model Architecture

### Design rationale

| Component | Choice | Why |
|---|---|---|
| Conv filters | 32 → 64 → 128 → 256 | Progressive feature abstraction |
| Batch Norm | After every Conv | Stable gradients, faster convergence |
| Dropout | 0.3 (conv), 0.5 (dense) | Regularisation against noisy inputs |
| Global Avg Pool | Instead of Flatten | Fewer parameters, less overfitting |
| L2 regularisation | Dense layers | Extra weight penalty |
| Activation | ReLU throughout | Standard, proven for vision tasks |

In [ ]:
def conv_block(x, filters, kernel_size=3, strides=1, dropout_rate=0.3):
    """Conv → BN → ReLU → optional Dropout block."""
    x = layers.Conv2D(
        filters, kernel_size,
        strides=strides,
        padding='same',
        kernel_initializer='he_normal',
        kernel_regularizer=regularizers.l2(1e-4),
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    if dropout_rate > 0:
        x = layers.SpatialDropout2D(dropout_rate)(x)
    return x


def build_improved_model(input_shape=(28, 28, 1), num_classes=10):
    """Improved CNN with BatchNorm, Dropout, and Global Average Pooling."""
    inputs = layers.Input(shape=input_shape)

    # ── Block 1: 28×28 → 14×14 ────────────────────────────────────────────
    x = conv_block(inputs, filters=32, dropout_rate=0.0)
    x = conv_block(x,      filters=32, dropout_rate=0.2)
    x = layers.MaxPooling2D((2, 2))(x)

    # ── Block 2: 14×14 → 7×7 ─────────────────────────────────────────────
    x = conv_block(x, filters=64, dropout_rate=0.0)
    x = conv_block(x, filters=64, dropout_rate=0.3)
    x = layers.MaxPooling2D((2, 2))(x)

    # ── Block 3: 7×7 → 3×3 ───────────────────────────────────────────────
    x = conv_block(x, filters=128, dropout_rate=0.0)
    x = conv_block(x, filters=128, dropout_rate=0.3)
    x = layers.MaxPooling2D((2, 2))(x)

    # ── Block 4 ───────────────────────────────────────────────────────────
    x = conv_block(x, filters=256, dropout_rate=0.0)
    x = conv_block(x, filters=256, dropout_rate=0.0)

    # ── Classifier head ───────────────────────────────────────────────────
    x = layers.GlobalAveragePooling2D()(x)          # avoids giant flatten
    x = layers.Dense(
        256, activation='relu',
        kernel_initializer='he_normal',
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(
        128, activation='relu',
        kernel_initializer='he_normal',
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs, name='improved_mnist_cnn')


model = build_improved_model()
model.summary()

## 5. Training Strategy

| Hyperparameter | Baseline | Improved |
|---|---|---|
| Optimiser | Adam 1e-4 | Adam 1e-3 with cosine decay |
| Batch size | 16 | 64 |
| Epochs | 4 | Up to 40 (early stopping) |
| LR schedule | None | ReduceLROnPlateau |
| Early stopping | None | patience=8, restore best weights |

In [ ]:
# ── Compile ───────────────────────────────────────────────────────────────────
INITIAL_LR = 1e-3

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=INITIAL_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ─────────────────────────────────────────────────────────────────
cb_early_stop = callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.4,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

cb_checkpoint = callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print("Compiled. Training starting...")

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS = 40

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=[cb_early_stop, cb_reduce_lr, cb_checkpoint],
    verbose=1
)

print("\nTraining complete.")
best_val_acc = max(history.history['val_accuracy'])
print(f"Best validation accuracy: {best_val_acc:.4f}")

## 6. Training Curves

In [ ]:
hist = history.history
epochs_ran = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Accuracy
axes[0].plot(epochs_ran, hist['accuracy'],     label='Train', lw=2, color='royalblue')
axes[0].plot(epochs_ran, hist['val_accuracy'], label='Val',   lw=2, color='tomato', linestyle='--')
axes[0].set_title('Accuracy',  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(epochs_ran, hist['loss'],     label='Train', lw=2, color='royalblue')
axes[1].plot(epochs_ran, hist['val_loss'], label='Val',   lw=2, color='tomato', linestyle='--')
axes[1].set_title('Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Learning rate
lrs = hist.get('lr', [INITIAL_LR] * len(epochs_ran))
axes[2].plot(epochs_ran, lrs, lw=2, color='seagreen')
axes[2].set_title('Learning Rate', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.suptitle('Training Curves — Improved Model', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Evaluation on Test Set

In [ ]:
# ── Load best saved weights ───────────────────────────────────────────────────
model.load_weights('best_model.keras')

# ── Final test evaluation ────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")

# Baseline for reference
BASELINE_ACC = 0.8730
improvement  = (test_acc - BASELINE_ACC) * 100
print(f"\nBaseline accuracy : {BASELINE_ACC*100:.2f}%")
print(f"Improved accuracy : {test_acc*100:.2f}%")
print(f"Delta             : {improvement:+.2f} percentage points")

In [ ]:
# ── Predictions ───────────────────────────────────────────────────────────────
y_pred_probs = model.predict(test_ds, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ['d', '.2f'],
    ['Confusion Matrix (counts)', 'Confusion Matrix (normalised)']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=range(10), yticklabels=range(10),
                linewidths=0.5, ax=ax)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True',      fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Per-class classification report ───────────────────────────────────────────
print("Per-class Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))

In [ ]:
# ── Per-class accuracy bar chart ──────────────────────────────────────────────
per_class_acc = cm_norm.diagonal()

fig, ax = plt.subplots(figsize=(10, 5))
colors  = ['#e74c3c' if v < 0.90 else '#2ecc71' for v in per_class_acc]
bars    = ax.bar(range(10), per_class_acc * 100, color=colors, edgecolor='white', linewidth=0.8)
ax.axhline(100 * per_class_acc.mean(), color='steelblue', linestyle='--', lw=1.5,
           label=f'Mean: {100*per_class_acc.mean():.1f}%')
ax.axhline(BASELINE_ACC * 100, color='orange', linestyle=':', lw=1.5,
           label=f'Baseline: {BASELINE_ACC*100:.1f}%')
for bar, val in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Per-class Test Accuracy', fontsize=13, fontweight='bold')
ax.set_xticks(range(10))
ax.set_ylim(0, 110)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Error Analysis — Most Confused Samples

In [ ]:
# Show the 20 most confident mistakes
errors      = np.where(y_pred != y_test)[0]
confidences = y_pred_probs[errors, y_pred[errors]]   # confidence in wrong answer
top_errors  = errors[np.argsort(confidences)[::-1][:20]]

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for i, (ax, idx) in enumerate(zip(axes.flat, top_errors)):
    img  = X_test_norm[idx, :, :, 0]
    true = int(y_test[idx])
    pred = int(y_pred[idx])
    conf = float(y_pred_probs[idx, pred])
    ax.imshow(img, cmap='gray')
    ax.set_title(f"True: {true}  Pred: {pred}\nConf: {conf:.2f}",
                 fontsize=9,
                 color='red' if true != pred else 'green')
    ax.axis('off')

plt.suptitle('Top-20 Most Confident Errors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Total errors : {len(errors)} / {len(y_test)}  ({len(errors)/len(y_test)*100:.2f}%)")

## 9. Final Summary

In [ ]:
print("=" * 55)
print("           FINAL RESULTS SUMMARY")
print("=" * 55)
print(f"  Baseline test accuracy   : {BASELINE_ACC*100:.2f}%")
print(f"  Improved test accuracy   : {test_acc*100:.2f}%")
print(f"  Improvement              : {improvement:+.2f} pp")
print()
print("  Key changes made:")
print("  • Deeper CNN: 32→64→128→256 filters with dual conv blocks")
print("  • Batch Normalisation after every Conv layer")
print("  • SpatialDropout2D (conv) + Dropout (dense) for regularisation")
print("  • Global Average Pooling instead of Flatten")
print("  • L2 weight regularisation on dense layers")
print("  • Data augmentation: rotation, shift, zoom, flip")
print("  • Z-score normalisation (train statistics only)")
print("  • Adam lr=1e-3 with ReduceLROnPlateau (factor=0.4, patience=3)")
print("  • EarlyStopping (patience=8, restore best weights)")
print("  • Batch size 64 for more stable gradient estimates")
print("=" * 55)